<center>
    <h2 style="font-size: 48px; margin-bottom: 5px;"><b>PROJECT QUIZ </b></h2>
</center>

<center>
    <h3 style="font-size: 36px; margin-top: 0px; margin-bottom: 5px;">- DELIVERABLE  -</h3>
</center>

<br> <br>

<center> <img src="https://www.alsacetech.org/wp-content/uploads/2017/05/cesi_CMJN-e1673249263164.png" alt="logo" width="200"></center>

<br> <br>

## Group 2 :
- ACACHA Elie
- PALE Kharice
- DEMIRHINDI Sude
- MERKICHE Isak
- LOUACHE Manyl

<br> <br>


# **CONTEXT**

Since the 1990s reducing energy consumption and greenhouse gas emissions have become a global objective. Transportation and delivery services contribute to these emissions improving delivery routes can help reduce the environmental impact. In this context ADEME (the French Environment and Energy Management Agency) promotes the development of mobility solutions for people and goods. CesiCDP wants to respond to this initiative by developing a method for managing delivery routes. Optimizing delivery routes can help companies reduce fuel consumption, travel time and operating costs.

# **PROBLEM**

Here we want to develop a method or a way to manage delivering route. Our main goal is to optimize those routes to reduces the company fuel comsumption, travel time and operating costs.The main problem of this project is to optimize the delivery routes on a road network. The objective is to find the efficient delivery routes for a convoy of delivering trucks that go to different customers and return to the starting point while minimizing travel time and respecting each delivery time window. This problem is known as the Capacitated Vehicle Routing Problem with Time Window (CVRPTW). Each truck has a capacity, and every customer must receive their deliveries in their time window and at each customer we must verify a certain service time duration. The algorithm must decide which customers each truck should visit and in which order. The main challenges are the number of possible deliveries routes the capacity limits of the trucks and the differences in traffic conditions and the differents time constraints. As the number of customers increases finding the best delivery routes becomes more difficult. This problem will the requires efficient algorithms.

# **OVERALL OBJECTIVES**
The main objective is to develop an algorithm of finding efficient delivery routes for realistic situations that involve several thousand customers.
The project has specific objectives:
- Model the delivery problem mathematically.
-	Develop an algorithm in Python to calculate delivery routes.
-	Respect the capacity limits of the trucks.
-	Minimize the travel time.
-	Evaluate the algorithms performance in terms of computation time and solution quality.
-	Test the algorithm on instances and compare its results with known reference solutions.
-	Time windows: each customer must be served during a specific period

# **Concrete Examples and Use Cases**

A typical example is a delivery company that must deliver packages to 100 customers using trucks. Each truck has a carrying capacity and every customer must be visited. A simple solution would be to visit customers in the order in which their addresses are received. However, this could create delivery routes, unnecessary travel and higher fuel consumption. With delivery route optimization the algorithm assigns customers to trucks and determines the order of visits to reduce the total travel time while respecting vehicle capacities. Another example is supermarket delivery service. Some customers may only be available to receive their orders between 10 a.m. And noon. In the version the algorithm must organize the delivery routes so that deliveries take place within these time windows. These applications show how delivery route optimization can improve delivery efficiency, reduce operating costs and support sustainable transportation.


# **Modeling and solution methods**


## Data Formalization

### 1. Benchmark instances and file formats

We test our methods on the **Solomon benchmark** (100 customers) and its extension by **Gehring & Homberger** (200 to 1000 customers), the standard benchmarks for the CVRPTW. All files come from the [PyVRP/VRPLIB](https://github.com/PyVRP/VRPLIB) repository. We read them with the Python package `vrplib`, which returns every instance as the same Python dictionary, whatever its file format.

The instances come in two file formats.

#### 1.1 Solomon format (`.txt`)

This is the original format of the benchmark. The file has a short header with the fleet, followed by a table with one row per node. Row `0` is the depot.

```
C101

VEHICLE
NUMBER     CAPACITY
  25         200

CUSTOMER
CUST NO.  XCOORD.   YCOORD.    DEMAND   READY TIME  DUE DATE   SERVICE   TIME
    0      40         50          0          0       1236          0
    1      45         68         10        912        967         90
    2      45         70         30        825        870         90
```

| Column | Meaning |
|---|---|
| `NUMBER` | number of available vehicles |
| `CAPACITY` | capacity of each vehicle (all vehicles are identical) |
| `CUST NO.` | node index (0 = depot) |
| `XCOORD.`, `YCOORD.` | coordinates of the node |
| `DEMAND` | quantity to deliver to the customer |
| `READY TIME`, `DUE DATE` | time window: earliest and latest start of service |
| `SERVICE TIME` | duration of the service at the customer |

The file contains **no distances**: `vrplib` computes them from the coordinates as real-valued Euclidean distances. It is read with `vrplib.read_instance(path, instance_format="solomon")`.

#### 1.2 VRPLIB / LKH-3 format (`.vrptw`)

This format follows the TSPLIB convention: a header of `KEY : value` lines, then one section per type of data.

```
NAME : C102
TYPE : CVRPTW
DIMENSION : 101          <- number of nodes (depot + 100 customers)
VEHICLES : 10
CAPACITY : 200
SERVICE_TIME : 90        <- one value for every customer
EDGE_WEIGHT_TYPE : EXACT_2D
NODE_COORD_SECTION       <- "node x y"
1 40 50
2 45 68
...
DEMAND_SECTION           <- "node demand"
TIME_WINDOW_SECTION      <- "node ready due"
DEPOT_SECTION            <- depot index, ended by -1
1
-1
EOF
```

Differences with the Solomon format:

- Nodes are numbered from **1** in the file; `vrplib` shifts them so that the depot is node **0**, as in the Solomon files.
- The service time is a **single value** for all customers. The depot has no service, so we set $s_0 = 0$.
- `EDGE_WEIGHT_TYPE` tells how distances are computed. With `EXACT_2D` and `EUC_2D`, `vrplib` returns the real Euclidean distance; with `FLOOR_2D`, it rounds the distance down to an integer. All the instances we use give real distances.
- In the LKH-3 files, the fleet is much tighter. In the file name `C102.100.10`, 100 is the number of customers and 10 the number of vehicles. Here $|K| = \lceil \sum_i d_i / Q \rceil$, so no vehicle is spare. The `.vrp` file of RC101 uses the same format but keeps the original Solomon fleet of 25 vehicles.

It is read with `vrplib.read_instance(path)`.

#### 1.3 Data actually used in our computations

Some fields of the files describe the instance but do not take part in the computations (`NAME`, `TYPE`, `COMMENT`, `DIMENSION`, `DEPOT_SECTION`, `SCALE`). The table below lists the fields that do take part, and how `vrplib` gives them to us.

| Data | Solomon `.txt` | `.vrptw` | `vrplib` key | Python shape | Symbol |
|---|---|---|---|---|---|
| Number of vehicles | `NUMBER` | `VEHICLES` | `vehicles` | integer | $\lvert K\rvert$ |
| Vehicle capacity | `CAPACITY` | `CAPACITY` | `capacity` | integer | $Q$ |
| Coordinates | `XCOORD.`, `YCOORD.` | `NODE_COORD_SECTION` | `node_coord` | $(n+1)\times 2$ | $(x_i, y_i)$ |
| Demands | `DEMAND` | `DEMAND_SECTION` | `demand` | $n+1$ | $d_i$ |
| Time windows | `READY TIME`, `DUE DATE` | `TIME_WINDOW_SECTION` | `time_window` | $(n+1)\times 2$ | $[b_i, e_i]$ |
| Service times | `SERVICE TIME` | `SERVICE_TIME` | `service_time` | $n+1$ or scalar | $s_i$ |
| Travel times | computed | computed | `edge_weight` | $(n+1)\times(n+1)$ | $\tau_{ij}$ |

The depot row gives $d_0 = 0$, and its time window $[b_0, e_0]$ is the working day: vehicles leave the depot at $b_0$ at the earliest and must be back by $e_0$.

#### 1.4 Instances used in this project

| Instance | Format | Customers $n$ | Vehicles $\lvert K\rvert$ | $Q$ | Total demand | Min. vehicles $\lceil \sum d_i / Q\rceil$ | $[b_0, e_0]$ | Used for |
|---|---|---|---|---|---|---|---|---|
| C101 | Solomon `.txt` | 100 | 25 | 200 | 1810 | 10 | [0, 1236] | ILP and ALNS, reference solution `C101.sol` |
| RC101 (first 20 customers) | `.vrp` (EUC_2D) | 20 | 25 | 200 | 430 | 3 | [0, 240] | ILP, tighter time windows |
| C102.100.10 | `.vrptw` (EXACT_2D) | 100 | 10 | 200 | 1810 | 10 | [0, 1236] | ALNS, wider time windows |
| C1_2_1 | Homberger `.txt` | 200 | 50 | 200 | 3530 | 18 | [0, 1351] | ALNS, reference solution `C1_2_1.sol` |

For the ILP, an instance can be cut down to its depot and its **first $n$ customers**, to study how the solving time grows with $n$.

For the reference instances, the `.sol` files give the best known solution: one line per route, then its cost. The published cost of C101 (827.3) uses distances truncated to one decimal, the original Solomon convention. With exact distances, the same routes cost **828.94**, so we always recompute the reference cost with our own distances before we compare.

### 2. Graph model

An instance is modelled as a **complete directed graph**

$$
G = (V, A), \qquad V = \{0\} \cup N, \qquad N = \{1, \dots, n\}, \qquad A = \{(i,j) \in V \times V : i \neq j\}
$$

- Node $0$ is the depot, where every route starts and ends.
- $N$ is the set of the $n$ customers.
- An arc $(i,j)$ is a direct trip from $i$ to $j$. Every pair of nodes is linked, so $\lvert A\rvert = n(n+1)$, which gives 650 arcs for 25 customers and 10 100 for 100 customers.

**Weight of an arc.** Following the Solomon convention, one unit of distance takes one unit of time, so the travel time is the Euclidean distance between the two nodes:

$$
\tau_{ij} = c_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}
$$

The travel time $\tau_{ij}$ is also the cost $c_{ij}$ that we minimise. The matrix is symmetric ($\tau_{ij} = \tau_{ji}$) and satisfies the triangle inequality, so going through an extra customer never shortens a trip.

**Weight of a node.** Each customer $i \in N$ carries a demand $d_i$, a time window $[b_i, e_i]$ and a service time $s_i$.

**Real-valued times.** Both methods use the exact, real Euclidean distances. The ILP uses the matrix `edge_weight` from `vrplib`, and the ALNS recomputes the same distances from the coordinates. In the ILP, the service start times are continuous variables, so travel times do not need to be integers.

### 3. Fleet

The fleet $K$ is **homogeneous**: every vehicle has the same capacity $Q = 200$ in all our instances. $\lvert K\rvert$ is an **upper bound**: a solution may use fewer vehicles. Each vehicle used performs a single route $R_k = (0, i_1, \dots, i_r, 0)$, and its load must not exceed the capacity:

$$
\sum_{i \in R_k} d_i \le Q
$$

Since demands cannot be split, at least $\lceil \sum_{i \in N} d_i / Q \rceil$ vehicles are needed. For C101 this gives 10 vehicles, and the best known solution indeed uses 10 routes out of the 25 available.

### 4. Time along a route

A vehicle leaves the depot at $T_0 = b_0$. When it goes from $i$ to $j$, it arrives at $a_j = T_i + s_i + \tau_{ij}$. If it arrives before the window opens, it waits, so service starts at

$$
T_j = \max(b_j, a_j)
$$

The route is feasible if $T_j \le e_j$ for every customer visited and if the vehicle is back at the depot by $e_0$. In C101 every customer needs $s_i = 90$ time units of service, which is long compared with the travel times (for example $\tau_{01} \approx 18.7$). Time windows, more than distances, therefore drive the number of customers a vehicle can serve.


## Complexity Proof
## 1. Complexity classification

The **Capacitated Vehicle Routing Problem with Time Windows (CVRPTW)** is an NP-hard combinatorial problem.

The objective is to minimize the total routing time cost while satisfying vehicle capacity constraints and customer time-window constraints. Every customer must be visited exactly once, and every vehicle route must start and end at the depot.

To establish its complexity, we consider the associated decision problem.

**CVRPTW-Decision**: Given a CVRPTW instance and a cost bound $B$, does there exist a set of feasible vehicle routes whose total cost is at most $B$ ?
The complexity classification is:
- Decision version: **NP-complete**.
- Optimization version: **NP-hard**.

We establish this classification by proving that the decision version belongs to NP and is NP-hard through a polynomial-time reduction from the decision version of the traveling salesman problem (TSP).

## 2. Proof that CVRPTW-Decision belongs to NP

To prove that CVRPTW-Decision belongs to NP, we must show that a proposed solution can be verified in polynomial time.

### Certificate

A certificate is a set of vehicle routes representing a solution structure for our problem:

$$
S=(R_1,R_2,\ldots,R_m)
\qquad m\leq |K|
$$

where each route starts and ends at the depot and gives the sequence of customers visited by its vehicle. The certificate has polynomial size: it lists each customer once and the depot twice per route, so at most $n + 2\lvert K\rvert$ nodes.

### Verification

Given a certificate $S$, we must verify the following conditions based on the problem constraints.

1. **Customer coverage:** Every customer is visited exactly once.
2. **Route structure:** Every vehicle route starts and ends at the depot, and the number of routes does not exceed the available fleet size.
3. **Capacity constraints:** For each route $R_k$, the total demand of its customers does not exceed the vehicle capacity $Q$.
4. **Time-window constraints:** Each customer is served within its time window.
5. **Cost bound:** the total cost of all routes is at most $B$.
6. **Depot return deadline:** every route returns by time $e_0$.

For time-window verification, let customer $i$ have a time window $[b_i,e_i]$, service duration $s_i$, and let $\tau_{ij}$ be the travel time from customer $i$ to customer $j$.

The earliest feasible service-start time at customer $j$, following customer $i$, is:

$$
T_j=\max\left(b_j,T_i+s_i+\tau_{ij}\right)
$$

We then verify that $T_j\leq e_j$.
If this inequality fails for any customer, the proposed route is infeasible. Otherwise, we continue along the route, and finally check that the vehicle is back at the depot by $e_0$.

Each node of the certificate is processed once, in constant time: one look-up in the travel-time matrix, one update of the load and of the time, and one mark in a boolean array to detect a customer visited twice or never. The whole verification therefore runs in $O(n + \lvert K\rvert)$, which is polynomial in the input size.

Therefore,

$$
\boxed{\text{CVRPTW-Decision}\in NP}
$$

## 3. Proof that CVRPTW-Decision is NP-Hard

We are going to prove this by reducing the decision problem of the travelling salesman problem to CVRPTW-Decision.

### 3.1. Source problem: TSP-Decision

An instance of TSP-Decision consists of a complete weighted graph $G=(V,E)$
a depot vertex $r\in V$, edge costs $c_{ij}$ and a cost bound $B$.

The question is whether there exists a Hamiltonian cycle that starts at $r$, visits every vertex exactly once, returns to $r$, and has total time cost at most $B$.

TSP-Decision is NP-complete (Karp, 1972, through the Hamiltonian cycle problem).

### 3.2. Polynomial-time transformation

Let $m=|V|$. Given an arbitrary TSP-Decision instance, construct a CVRPTW instance as follows:

- **Depot:** use the TSP vertex $r$ as the depot, relabeled $0$.
- **Customers:** each vertex in $V\setminus\{r\}$ becomes a customer.
- **Fleet size:** provide exactly **one** vehicle.
- **Demand:** set $d_i=1$ for every customer.
- **Vehicle capacity:** set $Q=m-1$, sufficient to serve all customers.
- **Travel costs:** preserve every TSP arc cost, $c'_{ij}=c_{ij}$.
- **Travel times:** set $\tau'_{ij}=c_{ij}$ so that total travel time equals TSP tour cost.
- **Service times:** set $s_i=0$ for every customer and for the depot.
- **Time windows:** assign $[0,H]$ to every customer and to the depot, with

$$
H=mC_{\max}+1,\qquad C_{\max}=\max_{(i,j)\in E}c_{ij}.
$$

A Hamiltonian cycle has exactly $m$ arcs, each costing at most $C_{\max}$; therefore its total travel time is at most $mC_{\max}<H$. These wide windows do not remove any TSP tour.

- **Objective bound:** keep the same bound $B$.

The constructed instance contains a polynomial number of nodes and arcs. Because the integer $H$ has polynomially many bits relative to the input encoding, this transformation runs in polynomial time.

### 3.3. Proof of reduction

We prove equivalence in both directions.

**Direction 1: TSP-Decision has a solution $\Rightarrow$ CVRPTW-Decision has a solution.**

Suppose that the TSP instance has a Hamiltonian cycle of total cost at most $B$. Use this cycle as the route of the single vehicle. Every customer is visited exactly once; capacity $Q=m-1$ allows all unit demands; service durations are zero; and the time horizon is large enough to accommodate the complete route. Since travel times equal the original TSP costs, the total CVRPTW travel time is also at most $B$. Therefore, the constructed CVRPTW instance is a **yes-instance**.

**Direction 2: CVRPTW-Decision has a solution $\Rightarrow$ TSP-Decision has a solution.**

Suppose that the constructed CVRPTW instance admits a feasible solution of total travel time at most $B$. There is only one vehicle, and the routing rules require it to start from the depot, serve every customer exactly once and return to the depot. Consequently, its route is a Hamiltonian cycle through all vertices of the original TSP graph. Because the travel times were copied from TSP arc costs, the original cycle has cost at most $B$. Therefore, the TSP instance is also a **yes-instance**.

We have established:

$$
\boxed{\text{TSP-Decision is a yes-instance}\iff\text{CVRPTW-Decision is a yes-instance}.}
$$

Hence:

$$
\text{TSP-Decision}\leq_p\text{CVRPTW-Decision}.
$$

Since TSP-Decision is NP-complete, CVRPTW-Decision is NP-hard. Combined with membership in NP, this yields:

$$
\boxed{\text{CVRPTW-Decision is NP-complete}.}
$$

Consequently, the **optimization version of the general CVRPTW is NP-hard**.


## Mathematical Modeling (ILP)

We use the compact **two-index formulation**. Since all vehicles are identical, one set of arc variables is enough for the whole fleet: we do not need to know *which* vehicle drives an arc, only *whether* an arc is used. Loads and times are tracked with Miller–Tucker–Zemlin (MTZ) style constraints, linearised with big-M.

### 1. Data

We use the notation of the Data Formalization: the graph $G = (V, A)$ with $V = \{0\} \cup N$, the travel times $\tau_{ij} = c_{ij}$, the demands $d_i$, the service times $s_i$ (with $s_0 = 0$), the time windows $[b_i, e_i]$, the depot window $[b_0, e_0]$, the capacity $Q$ and the fleet size $\lvert K\rvert$.

### 2. Variables

| Variable | Domain | Meaning |
|---|---|---|
| $x_{ij}$ | $\{0,1\}$, $(i,j) \in A$ | $=1$ if a vehicle travels directly from $i$ to $j$ |
| $T_i$ | $\mathbb{R}_{\ge 0}$, $i \in V$ | start of service at node $i$ ($T_0$ = departure time from the depot) |
| $u_i$ | $\mathbb{R}$, $i \in N$ | load of the vehicle just after it serves customer $i$ |

### 3. Objective

$$
\min \sum_{(i,j) \in A} c_{ij}\, x_{ij}
$$

### 4. Constraints

$$
\begin{aligned}
&(1)\ \textstyle\sum_{j : (i,j) \in A} x_{ij} = 1 && \forall i \in N && \text{each customer is left exactly once}\\
&(2)\ \textstyle\sum_{j : (j,i) \in A} x_{ji} = 1 && \forall i \in N && \text{each customer is reached exactly once}\\
&(3)\ \textstyle\sum_{j \in N} x_{0j} \le \lvert K\rvert && && \text{fleet size}\\
&(3')\ \textstyle\sum_{j \in N} x_{0j} \ge \left\lceil \sum_{i \in N} d_i / Q \right\rceil && && \text{valid inequality}\\
&(4)\ u_j \ge u_i + d_j - Q\,(1 - x_{ij}) && \forall (i,j) \in A,\ i, j \in N && \text{load propagation}\\
&(5)\ d_i \le u_i \le Q && \forall i \in N && \text{capacity}\\
&(6)\ T_j \ge T_i + s_i + \tau_{ij} - M_{ij}\,(1 - x_{ij}) && \forall (i,j) \in A,\ j \in N && \text{schedule consistency}\\
&(7)\ T_i + s_i + \tau_{i0} \le e_0 + M_{i0}\,(1 - x_{i0}) && \forall i \in N && \text{return to the depot before closing}\\
&(8)\ b_i \le T_i \le e_i && \forall i \in V && \text{time windows}
\end{aligned}
$$

Constraints (1) and (2) also balance the depot: $n$ arcs leave the customers and $n$ arcs reach them, so as many vehicles come back to the depot as leave it. All vehicles share the departure time $T_0$. This loses nothing: waiting is allowed, so leaving at $b_0$ is always possible.

### 5. Choice of the big-M values

When $x_{ij} = 1$, the term $M(1 - x_{ij})$ is zero and the constraint applies. When $x_{ij} = 0$, the constraint must hold for **every** feasible value of the variables, so that it is inactive. This gives the **smallest valid M**:

- **Time (6):** we need $T_i + s_i + \tau_{ij} - M_{ij} \le T_j$ for all $T_i \le e_i$ and $T_j \ge b_j$, so
$$M_{ij} = \max(0,\ e_i + s_i + \tau_{ij} - b_j).$$
- **Return (7):** in the same way, $M_{i0} = \max(0,\ e_i + s_i + \tau_{i0} - e_0)$.
- **Load (4):** we need $u_i + d_j - M \le u_j$ for all $u_i \le Q$ and $u_j \ge d_j$, so $M = Q$ is enough.

Any larger M gives the same integer solutions, but a weaker **linear relaxation**: a small fractional value of $x_{ij}$ is then enough to switch the constraint almost off. The lower bound gets worse, and the branch and bound must explore more nodes before it proves optimality.

### 6. Subtour elimination

Constraints (4) and (6) also remove **subtours**, that is, cycles of customers that do not go through the depot. Along such a cycle, the load would increase strictly at every arc ($d_j > 0$) and so would the time ($s_i + \tau_{ij} > 0$), which is impossible on a cycle. This argument needs positive demands or positive times, which holds in every Solomon instance ($d_i > 0$ and $s_i > 0$). The depot is left out of (4) and handled by (7), so cycles through the depot, which are the actual routes, stay allowed.

### 7. Valid inequality (3')

At least $\lceil \sum_i d_i / Q \rceil$ vehicles are needed to carry the total demand, so (3') removes no integer solution. It does tighten the linear relaxation, though. Without it, when the objective is the number of vehicles, CBC quickly finds a solution with the minimum fleet but struggles to **prove** that no smaller fleet exists.

### 8. Preprocessing

Before building the model, we remove the arcs between two customers that no feasible solution can use:

- **Time:** even when service at $i$ starts as early as possible, the vehicle reaches $j$ too late, that is $b_i + s_i + \tau_{ij} > e_j$.
- **Capacity:** the two demands together exceed the capacity, that is $d_i + d_j > Q$.

The arcs to and from the depot are always kept. The model gets smaller and the big-M values tighter. On C101 (100 customers), this leaves 4 512 arc variables out of 10 100, and 9 026 constraints.

### 9. Size of the model

The model has at most $n(n+1)$ binary variables, $2n + 1$ continuous variables and $O(n^2)$ constraints, so its size grows only quadratically. The difficulty lies elsewhere: the number of nodes explored by the branch and bound can grow exponentially with $n$, because the problem is NP-hard.


## Solution

a presentation of the potential applications of this exact approach (direct solution, benchmark, lower/upper bounds, etc.),

as well as:

a description of the chosen heuristic or metaheuristic solution method,

its formalization adapted to the problem, including, for example:

definition of solutions,

neighborhood structure (local search, VNS, LNS, etc.),

specific operators (crossover, mutation, insertion, exchange…),

guidance mechanisms (e.g., pheromones for ACO, speeds for PSO, etc.).

# **Implementation**

## Pseudo-Code (explaining our algorithm which ALNS)

## different stages 

## Implementation


## Test

# **Experimental Study**

